# Playground — `<modelo>` (`<seu nome>`)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpeuzinCosta/kws-borda/blob/main/notebooks/experimentos/playground_modelo.ipynb)

**Playground que usa os módulos do repositório.** Dados, MFCC, normalização, SpecAugment na GPU, treino,
avaliação, latência e a varredura de configurações vêm de `dados/`, `treino/` e `avaliacao/`, as mesmas funções
do notebook principal e do playground da CNN 1D. Aqui você escreve **só o modelo** e as configurações que quer
testar.

Como usar:
1. Abra no Colab (Ambiente de execução → Alterar tipo → GPU). Para guardar as alterações:
   **Arquivo → Salvar uma cópia no GitHub** ou baixe o `.ipynb` e substitua este arquivo.
2. Implemente o modelo na seção **4. Modelo** (os pontos marcados com `TODO`). O decorador
   `@registrar_modelo` faz a classe do notebook **substituir** o arquivo de `modelos/` nesta sessão: o treino, a
   latência e a varredura passam a usá-la sem copiar nada.
3. Use `fracao < 1` nos hiperparâmetros para iterar rápido; para números de verdade, `fracao = 1.0` e 3 seeds.
4. Quando a arquitetura estiver definida, siga a seção **8. Levando para o repositório**.

O setup clona/atualiza o repositório: o que estiver no GitHub (branch `main`) é o que roda.

## 1. Setup

No Colab, o Google Drive é montado: o `.tar.gz` do GSC, o cache de MFCC e os treinos ficam em
`MyDrive/kws-borda/` e sobrevivem ao fim da sessão (são os mesmos do notebook principal).

In [ ]:
#@title Setup — rode esta célula primeiro
#Funciona no Colab (clona o repo e usa o Drive) e localmente/VS Code (usa a pasta do repositório).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/gpeuzinCosta/kws-borda.git"
NO_COLAB = "google.colab" in sys.modules

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO = Path("/content/kws-borda")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)     #pega a versão mais nova dos módulos
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

    #Drive: cache de MFCC, runs e resultados PERSISTEM entre sessões
    #Disco local (/content/data): áudios extraídos, leitura rápida (o Drive é lento para 100 mil arquivos)
    os.environ["KWS_DRIVE"] = "/content/drive/MyDrive/kws-borda"
    os.environ["KWS_DATA"] = "/content/data"
else:
    #Local: sobe a partir da pasta atual até achar a raiz do repositório
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())

#A raiz do repo no sys.path deixa importar as pastas: configs, dados, modelos, treino, avaliacao
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
from configs import pastas_projeto, escolher_device

PASTAS = pastas_projeto()
DOWNLOADS = Path(os.environ.get("KWS_DRIVE", REPO)) / "downloads"     #.tar.gz/.zip baixados UMA vez
device = escolher_device()
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

print(f"Colab: {NO_COLAB} | device: {device} | commit do código: {commit}")
print(f"torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—'}")
for nome, p in PASTAS.items():
    print(f"  {nome:10s} -> {p}")

## 2. Hiperparâmetros

Vêm de `configs/__init__.py`: `HYPERPARAMETERS` (regras comuns a todos) + `HP_MODELOS[<modelo>]` (só o que muda).
Aqui ficam só as sobrescritas deste playground.

In [ ]:
from configs import carregar_config

NOME_MODELO = "dscnn"     #troque pelo seu: mlp, cnn1d, resnet, dscnn ou atencao (precisa existir em HP_MODELOS, configs/__init__.py)

hyperparameters = carregar_config(NOME_MODELO,
                                  fracao=0.2,      #fração do TREINO (1.0 = tudo). Val e teste são sempre completos
                                  epocas=20,
                                  seeds=[0])       #[0, 1, 2] para os números de verdade
hyperparameters

## 3. Dados

O GSC v2 é baixado para o Drive **uma vez**, e o MFCC de treino/val/teste é calculado uma vez e guardado em
cache (`carregar_dados`). O `criar_dataloaders` normaliza com as estatísticas do treino e entrega o batch inteiro
com um fatiamento; o SpecAugment é aplicado depois, no batch, na GPU, dentro do treino.

In [ ]:
import subprocess, sys
from dados import classes_da_tarefa, criar_dataloaders
from treino.varredura import carregar_dados

subprocess.run([sys.executable, str(REPO / "dados" / "baixar.py"), "--dataset", "gsc",
                "--destino", str(PASTAS["dados"]), "--arquivos", str(DOWNLOADS)], check=True)
RAIZ_GSC = PASTAS["dados"] / "gsc"
CLASSES = classes_da_tarefa(hyperparameters["tarefa"])

dados = carregar_dados(hyperparameters, RAIZ_GSC, PASTAS["cache"])
loaders, (media, desvio) = criar_dataloaders(hyperparameters, dados)

xb, yb = next(iter(loaders["treino"]))
print("batch:", tuple(xb.shape), "| treino:", len(loaders["treino"].dataset), "áudios")   #esperado: (128, n_mfcc, n_frames)

## 4. Modelo

**Contrato** (o mesmo da bancada):
- entrada `x`: `(B, n_mfcc, n_frames)`, o MFCC normalizado (40 × 101 no padrão, 10 × 49 no `hello_edge`;
  não fixe esses números no código);
- saída: logits `(B, n_classes)`;
- a última camada se chama `self.head`;
- o `__init__` recebe `n_classes, n_mfcc, n_frames` e, depois, os hiperparâmetros da arquitetura (que vêm do
  `HP_MODELOS` em `configs/__init__.py` e das configurações da varredura).

`MeuModelo` abaixo é só um exemplo mínimo para o notebook rodar de ponta a ponta. Substitua pela sua
arquitetura, com os hiperparâmetros que ela precisar (ponha os valores em `HP_MODELOS[<modelo>]`, `configs/__init__.py`).

In [ ]:
import torch
from modelos import registrar_modelo


#O decorador registra a classe com o nome NOME_MODELO: nesta sessão ela substitui o arquivo modelos/<modelo>.py
@registrar_modelo(NOME_MODELO)
class MeuModelo(torch.nn.Module):
    """Exemplo mínimo (achata o MFCC + 1 camada escondida). Troque pela sua arquitetura."""

    def __init__(self, n_classes, n_mfcc=40, n_frames=101, hidden=64):
        super().__init__()

        self.corpo = torch.nn.Sequential(
            torch.nn.Flatten(),                           #(B, n_mfcc, n_frames) -> (B, n_mfcc * n_frames)
            torch.nn.Linear(n_mfcc * n_frames, hidden),
            torch.nn.ReLU(),
        )
        self.head = torch.nn.Linear(hidden, n_classes)    #última camada sempre com o nome `head`

    def forward(self, input_tensor):
        x = self.corpo(input_tensor)                      # (B, hidden)
        return self.head(x)                               # (B, 35)

In [ ]:
from modelos import criar_modelo
from avaliacao import custo

modelo = criar_modelo(NOME_MODELO, hyperparameters)
print("classe usada:", type(modelo).__name__, "| definida em:", type(modelo).__module__)   #'__main__' = a deste notebook

#Contrato
saida = modelo.eval()(xb)
print("entrada:", tuple(xb.shape), "| saída:", tuple(saida.shape))      #esperado: (128, 35)
assert saida.shape == (len(xb), len(CLASSES)), f"saída {tuple(saida.shape)}, esperado ({len(xb)}, {len(CLASSES)})"
assert hasattr(modelo, "head"), "a última camada deve se chamar `head`"

c = custo(modelo)
print(f"parâmetros: {c['params']:,} | {c['kb_int8']:.1f} KB int8 | MACs: {c['macs']:,}")

## 5. Treino e avaliação de um baseline

O loop único da bancada (`treino/loop.py`): AdamW, OneCycle, entropia cruzada com label smoothing, SpecAugment
no batch e os pesos da melhor época na validação.

In [ ]:
from treino import treinar
from avaliacao.graficos import curvas_treino

model_base, hist_base = treinar(criar_modelo(NOME_MODELO, hyperparameters), loaders, hyperparameters,
                                seed=hyperparameters["seeds"][0], device=device)
curvas_treino(hist_base, titulo=f"({NOME_MODELO})")
print(f'Gap (val - treino): {hist_base["gap"]:+.4f} | {hist_base["tempo_min"]:.1f} min')

In [ ]:
import pandas as pd
from treino import prever
from avaliacao import acuracia, f1_macro, matriz_confusao, top_confusoes, relatorio_por_classe
from avaliacao.graficos import plot_matriz_confusao

y_pred, y_true = prever(model_base, loaders["teste"])
print(f"teste | acurácia: {acuracia(y_pred, y_true)*100:.2f}% | F1 macro: {f1_macro(y_pred, y_true):.4f}")

cm = matriz_confusao(y_pred, y_true, len(CLASSES))
plot_matriz_confusao(cm, CLASSES, titulo=f"{NOME_MODELO} (teste)")
display(pd.DataFrame(top_confusoes(cm, CLASSES, k=10), columns=["verdadeira", "prevista", "fração"]))

#Ranking de F1 por classe na VALIDAÇÃO (do pior para o melhor)
y_pred_val, y_true_val = prever(model_base, loaders["val"])
display(relatorio_por_classe(y_pred_val, y_true_val, CLASSES).round(3))

## 6. Latência (MFCC × rede)

Protocolo da hipótese (b): CPU com **1 thread**, **batch 1**, aquecimento e mediana; MFCC, normalização e rede
medidos separados, em rodadas, com aviso se a medida oscilar.

In [ ]:
from avaliacao import medir_latencia

lat = medir_latencia(model_base, hyperparameters, media, desvio)
print(f"CPU: {lat['cpu']}")
print(f"MFCC: {lat['mfcc_ms']:.3f} ms | normalização: {lat['norm_ms']:.3f} ms | rede: {lat['rede_ms']:.3f} ms")
print(f"pré-processamento / rede = {lat['preproc_sobre_rede']:.2f}x | estável: {lat['estavel']}")

## 7. Varredura: quanto dá para encolher?

Cada configuração é `(nome, o que muda em relação ao hyperparameters)`. A primeira se chama `baseline`: é a
referência do critério do projeto (o **menor** modelo até 2 p.p. abaixo dela na **validação**).
Com `retreinar=False`, o que já foi treinado (pesos no Drive, tabela `runs/varredura_<modelo>.csv`) é carregado.

In [ ]:
from treino.varredura import (rodar_varredura, medir_latencias_varredura, anexar_latencias, montar_resumo,
                              tabela_slide, escolher_vencedor, matrizes_escolhidos)

#(nome, o que muda em relação ao hyperparameters). Troque pelas variações do seu modelo; a primeira é a referência
configs_varredura = [
    ('baseline',                 {}),
    ('sem spec_augment',         {'spec_augment': False}),
    ('hidden 32',                {'hidden': 32}),
    ('hidden 16',                {'hidden': 16}),
    ('entrada hello_edge',       {'n_mfcc': 10, 'janela_ms': 40, 'passo_ms': 20, 'center': False}),
    ('hello_edge + hidden 32',   {'n_mfcc': 10, 'janela_ms': 40, 'passo_ms': 20, 'center': False, 'hidden': 32}),
]

SEEDS = hyperparameters["seeds"]
tabela, modelos = rodar_varredura(NOME_MODELO, configs_varredura, hyperparameters, SEEDS, RAIZ_GSC, PASTAS,
                                  retreinar=False, device=device)

In [ ]:
from avaliacao.graficos import acuracia_vs_custo_varredura, barras_preproc_rede

latencias = medir_latencias_varredura(NOME_MODELO, configs_varredura, hyperparameters)
tabela = anexar_latencias(tabela, latencias)
resumo = montar_resumo(tabela, hyperparameters["fracao"], hyperparameters["margem_pp"])

display(tabela_slide(resumo))
acuracia_vs_custo_varredura(resumo, hyperparameters["margem_pp"])
barras_preproc_rede(resumo)

In [ ]:
vencedor = escolher_vencedor(resumo)

#Matrizes de confusão (no TESTE, somadas entre as seeds): baseline, vencedor e o que ficou logo abaixo da linha
matrizes_escolhidos(resumo, modelos, configs_varredura, hyperparameters, SEEDS, RAIZ_GSC, PASTAS["cache"])

## 8. Levando para o repositório

Quando a arquitetura estiver definida (o mesmo caminho da CNN 1D):
1. Copie a classe para `modelos/<modelo>.py`, no lugar do esqueleto, e mude `IMPLEMENTADO = True`.
2. Ponha a configuração vencedora em `HP_MODELOS[<modelo>]` (`configs/__init__.py`) e em `hp_modelos` no notebook
   principal (a arquitetura e, se a entrada mudar, o MFCC, como na `cnn1d` e na `resnet`).
3. Copie a lista `configs_varredura` para `treino/varredura.py` (ex.: `CONFIGS_DSCNN`), com o MFCC e a
   arquitetura do baseline explícitos em cada linha (o `HP_MODELOS` passa a guardar o vencedor, não o baseline).
4. Rode `pytest` para conferir o contrato.
5. No notebook principal, acrescente o estudo do modelo no formato da §8.1 (CNN 1D): só o baseline e o vencedor rodam lá; o resto fica referenciado no playground.

## Anotações

| Data | Mudança | Params | Acurácia val | Observação |
|---|---|---|---|---|
| | | | | |